# Feature Extraction Evaluation Pipeline

This notebook implements a complete workflow to test and evaluate LLM-based feature extraction against manually annotated ecological dataset metadata.

## Workflow Steps

### 1. Data Loading & Filtering
- Load the manually annotated dataset (`dataset_092624.xlsx`)
- Filter for Dryad source records with `valid_yn='yes'`
- Select rows where `_position` columns contain 'abstract' (indicating features were found in abstract)
- Select 5 test rows for evaluation

### 2. Data Validation
- Validate the manual annotations against the `DatasetFeatureExtraction` Pydantic schema
- Clean and normalize data using the `DataFrameValidator`

### 3. Automated Feature Extraction
- Extract abstracts from the `full_text` column
- Run GPT-based classification using `DatasetFeatureExtraction` schema
- Collect extraction results

### 4. Side-by-Side Comparison
- Build a comparison DataFrame with manual vs automated extractions
- Display results for visual inspection

### 5. Evaluation & Metrics
- Use `evaluation.py` utilities to compute precision, recall, F1 scores
- Normalize data between manual and automated sets as needed
- Generate evaluation report

---
**Test Dataset**: 5 Dryad records with abstract-derived annotations

### 1. Data Loading & Filtering


In [1]:
# Data exploration: load and inspect the dataset
import pandas as pd
import os

print("Current working directory:", os.getcwd())

# Use absolute path to avoid issues
ANNOT_FILE = r"data\dataset_092624.xlsx"
df = pd.read_excel(ANNOT_FILE)

print("Shape:", df.shape)
print("\nAll columns:")
print(df.columns.tolist())

Current working directory: c:\Users\beav3503\dev\llm_metadata
Shape: (418, 43)

All columns:
['id', 'url', 'title', 'full_text', 'publication_year', 'source', 'id_query', 'reason_non_valid', 'valid_yn', 'dataset_relevance', 'dataset_location', 'dataset_format', 'geospatial_info_dataset', 'geospatial_info_repo_page_text', 'geospatial_info_article_text', 'species', 'temporal_range', 'temp_range_i', 'temp_range_f', 'temporal_range_norm', 'temporal_range_position', 'temporal_duration_y', 'temporal_duration_position', 'spatial_range_km2', 'spatial_range_position', 'data_type', 'data_type_norm', 'referred_dataset', 'relevance_referred_dataset', 'journal', 'cited_articles', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'url.1', 'MC_dataset_type', 'MC_spatial_range', 'MC_temporal_range', 'MC_relevance', 'MC_relevance_modifiers']


In [2]:
# Explore _position columns and their values
position_cols = [c for c in df.columns if '_position' in c.lower()]
print("Position columns:", position_cols)

for col in position_cols:
    print(f"\n{col} unique values:")
    print(df[col].value_counts(dropna=False))

Position columns: ['temporal_range_position', 'temporal_duration_position', 'spatial_range_position']

temporal_range_position unique values:
temporal_range_position
NaN                                                       281
source publication text                                    86
source publication text, dataset                           12
not given                                                   9
source link abstract                                        7
source link abstract, dataset                               5
source link abstract, source publication text, dataset      4
source link                                                 4
source link abstract, source publication text               4
dataset                                                     2
source link abstract, dataset, source publication text      1
source link, dataset                                        1
dataset, source link                                        1
dataset, source publication 

In [3]:
# Filter: source='dryad', valid_yn='yes', and at least one _position column contains 'abstract'
print("Source values:", df['source'].value_counts())
print("\nvalid_yn values:", df['valid_yn'].value_counts())

# Filter for dryad and valid
dryad_valid = df[(df['source'] == 'dryad') & (df['valid_yn'] == 'yes')]
print(f"\nDryad valid rows: {len(dryad_valid)}")

# Find rows where any _position column contains 'abstract'
position_cols = ['temporal_range_position', 'temporal_duration_position', 'spatial_range_position']

def has_abstract_in_position(row):
    for col in position_cols:
        val = row.get(col)
        if pd.notna(val) and 'abstract' in str(val).lower():
            return True
    return False

dryad_abstract = dryad_valid[dryad_valid.apply(has_abstract_in_position, axis=1)]
print(f"Dryad valid rows with 'abstract' in _position columns: {len(dryad_abstract)}")

Source values: source
semantic_scholar    254
zenodo              114
dryad                47
referenced            3
Name: count, dtype: int64

valid_yn values: valid_yn
yes    299
no     119
Name: count, dtype: int64

Dryad valid rows: 37
Dryad valid rows with 'abstract' in _position columns: 11


In [4]:
# Check if there's an abstract column (full_text might be the abstract)
print("Checking full_text column (likely the abstract):")
print(f"Non-null full_text in dryad_abstract: {dryad_abstract['full_text'].notna().sum()}")
print("\nSample full_text (first 500 chars):")
print(dryad_abstract['full_text'].iloc[0][:500] if pd.notna(dryad_abstract['full_text'].iloc[0]) else "N/A")

Checking full_text column (likely the abstract):
Non-null full_text in dryad_abstract: 11

Sample full_text (first 500 chars):
Data from: The genetic signature of range expansion in a disease vector - the black-legged tick  Monitoring and predicting the spread of emerging infectious diseases requires that we understand the mechanisms of range expansion by its vectors. Here, we examined spatial and temporal variation of genetic structure among 13 populations of the Lyme disease vector, the black-legged tick, in southern Quebec, where this tick species is currently expanding and Lyme disease is emerging. Our objective was


In [5]:
dryad_abstract['url'].iloc[0]

'https://doi.org/10.5061/dryad.2n5h6'

In [6]:
# Show the relevant manual annotation columns that we'll compare with extraction
# These are features we'll extract and evaluate
relevant_cols = ['url', 'title', 'full_text', 'data_type', 'geospatial_info_dataset', 
                 'spatial_range_km2', 'temporal_range', 'temp_range_i', 'temp_range_f', 
                 'species', 'referred_dataset']

# Select 5 rows for testing
# For reproducibility, reuse same 5 from url

test_df = dryad_abstract.loc[[11, 15, 17, 18, 24], relevant_cols]

print(f"Selected {len(test_df)} rows for testing")
print("\nTest rows preview:")
test_df[['url', 'title', 'data_type', 'species']].head()

Selected 5 rows for testing

Test rows preview:


,url,title,data_type,species
11,https://doi.org/10.5061/dryad.2n5h6,Data from: The genetic signature of range expa...,"presence only, EBV genetic analysis",black-legged tick
15,https://doi.org/10.5061/dryad.3nh72,Data from: Patchy distribution and low effecti...,"presence only, EBV genetic analysis","Eastern coyote, eastern wolf"
17,https://doi.org/10.5061/dryad.4767v,Data from: Ecological gradients driving the di...,abundance,"Rhododendron groenlandicum, Kalmia angustifoli..."
18,https://doi.org/10.5061/dryad.4k275,"Data from: Caribou, water, and ice – fine-scal...",presence only,Rangifer tarandus
24,https://doi.org/10.5061/dryad.67t23,Data from: Severe recent decrease of adult bod...,EBV genetic analysis,Tachycineta bicolor


## Step 2: Validate Manual Annotations

In [7]:
%load_ext autoreload
%autoreload 2

In [8]:
from llm_metadata.schemas.validation import DataFrameValidator
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction

# Validate the test dataframe against the Pydantic schema
validator = DataFrameValidator(DatasetFeatureExtraction)
report = validator.validate(test_df)

print(report.summary())

if report.errors:
    print("\nSample Errors:")
    display(report.errors_to_dataframe())

2026-01-07 14:41:18.680 | INFO     | llm_metadata.schemas.validation:validate:233 - Validation complete: 5 valid, 0 invalid, 0 errors


VALIDATION REPORT
Total rows:     5
Valid rows:     5 (100.0%)
Invalid rows:   0

Error Breakdown:
  Type/Schema errors: 0
  Data/Value errors:  0

Errors by Field:


In [9]:
# Get validated manual annotations as Pydantic models
manual_models = report.valid_rows
manual_indices = report.valid_indices

# Create a mapping of DOI -> manual model for evaluation later
def extract_doi(url):
    return url.replace('https://doi.org/', '')

manual_by_doi = {}
for idx, model in zip(manual_indices, manual_models):
    doi = extract_doi(test_df.loc[idx, 'url'])
    manual_by_doi[doi] = model

print(f"Validated {len(manual_by_doi)} manual annotations")
print("DOIs:", list(manual_by_doi.keys()))

Validated 5 manual annotations
DOIs: ['10.5061/dryad.2n5h6', '10.5061/dryad.3nh72', '10.5061/dryad.4767v', '10.5061/dryad.4k275', '10.5061/dryad.67t23']


## Step 3: Automated Feature Extraction

Run GPT-based feature extraction on the abstracts using the `DatasetFeatureExtraction` schema.

In [10]:
from llm_metadata.gpt_classify import classify_abstract
from llm_metadata.schemas.fuster_features import DatasetFeatureExtraction

# Extract features from abstracts
automated_by_doi = {}

for idx in manual_indices:
    row = test_df.loc[idx]
    doi = extract_doi(row['url'])
    abstract = row['full_text']
    
    print(f"Processing DOI: {doi}")
    
    # Run GPT classification with DatasetFeatureExtraction schema
    result = classify_abstract(
        abstract=abstract,
        text_format=DatasetFeatureExtraction,
        model="gpt-5-mini",
        reasoning={"effort": "low"},
        max_output_tokens=4096,
    )
    
    automated_by_doi[doi] = result['output']
    print(f"  -> Extracted: {result['output'].data_type}")

print(f"\nCompleted extraction for {len(automated_by_doi)} abstracts")

Processing DOI: 10.5061/dryad.2n5h6
  -> Extracted: ['genetic_analysis', 'time_series', 'distribution']
Processing DOI: 10.5061/dryad.3nh72
  -> Extracted: ['genetic_analysis', 'distribution']
Processing DOI: 10.5061/dryad.4767v
  -> Extracted: ['abundance', 'distribution']
Processing DOI: 10.5061/dryad.4k275
  -> Extracted: ['presence-only', 'distribution', 'time_series']
Processing DOI: 10.5061/dryad.67t23
  -> Extracted: ['traits', 'abundance', 'time_series', 'ecosystem_function', 'distribution']

Completed extraction for 5 abstracts


## Step 4: Side-by-Side Comparison

Create a comparison DataFrame showing manual vs automated extractions for visual inspection.

In [11]:
# Build side-by-side comparison DataFrame
comparison_fields = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
                     'temporal_range', 'temp_range_i', 'temp_range_f', 'species']

comparison_rows = []

for doi in manual_by_doi.keys():
    manual = manual_by_doi[doi]
    auto = automated_by_doi[doi]
    
    # Add manual row
    manual_dict = manual.model_dump()
    manual_row = {'doi': doi, 'source': 'manual'}
    for field in comparison_fields:
        manual_row[field] = manual_dict.get(field)
    comparison_rows.append(manual_row)
    
    # Add automated row
    auto_dict = auto.model_dump()
    auto_row = {'doi': doi, 'source': 'automated'}
    for field in comparison_fields:
        auto_row[field] = auto_dict.get(field)
    comparison_rows.append(auto_row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(['doi', 'source'], ascending=[True, False])
comparison_df

,doi,source,data_type,geospatial_info_dataset,spatial_range_km2,temporal_range,temp_range_i,temp_range_f,species
0,10.5061/dryad.2n5h6,manual,"[presence-only, genetic_analysis]",[sample],2080.0,2011-2014,2011.0,2014.0,[black-legged tick]
1,10.5061/dryad.2n5h6,automated,"[genetic_analysis, time_series, distribution]","[sample, site, administrative_units]",NaN,2011 to 2014,2011.0,2014.0,"[black-legged tick, Lyme disease vector, Ixode..."
2,10.5061/dryad.3nh72,manual,"[presence-only, genetic_analysis]",[sample],NaN,2010-2014,2010.0,2014.0,"[Eastern coyote, eastern wolf]"
3,10.5061/dryad.3nh72,automated,"[genetic_analysis, distribution]","[sample, site_ids, geographic_features, admini...",NaN,2010\t\t6\t\t14,2010.0,2014.0,"[Canis lupus familiaris, Canis lycaon, eastern..."
4,10.5061/dryad.4767v,manual,[abundance],None,535355.0,1986-2000,1986.0,2000.0,"[Rhododendron groenlandicum, Kalmia angustifol..."
5,10.5061/dryad.4767v,automated,"[abundance, distribution]","[sample, maps, administrative_units, range, di...",535355.0,None,NaN,NaN,"[Rhododendron groenlandicum, Kalmia angustifol..."
6,10.5061/dryad.4k275,manual,[presence-only],[sample],630000.0,"1947-1985, 2007-2014",1947.0,1985.0,[Rangifer tarandus]
7,10.5061/dryad.4k275,automated,"[presence-only, distribution, time_series]","[sample, geographic_features, maps]",NaN,GPS telemetry 2007–2014; MODIS maps (8-day com...,1947.0,2014.0,"[migratory caribou, Rangifer tarandus caribou]"
8,10.5061/dryad.67t23,manual,[genetic_analysis],[site],10200.0,2005-2011,2005.0,2011.0,[Tachycineta bicolor]
9,10.5061/dryad.67t23,automated,"[traits, abundance, time_series, ecosystem_fun...",[administrative_units],NaN,2005–2011,2005.0,2011.0,"[Tree swallow (Tachycineta bicolor), aerial in..."


## Step 5: Evaluation & Metrics

Use the `evaluation.py` utilities to compute precision, recall, F1, and other metrics.

In [12]:
from llm_metadata.schemas.evaluation import (
    evaluate_indexed,
    EvaluationConfig,
    micro_average,
    macro_f1
)

# Run evaluation using the indexed dictionaries
eval_fields = ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 
               'temporal_range', 'temp_range_i', 'temp_range_f', 'species']

report = evaluate_indexed(
    true_by_id=manual_by_doi,
    pred_by_id=automated_by_doi,
    fields=eval_fields,
    config=EvaluationConfig(treat_lists_as_sets=True)
)

print("Per-field metrics:")
display(report.metrics_to_pandas())

Per-field metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,4,11,3,0,5,0.266667,0.571429,0.363636,0.8,0.0
1,geospatial_info_dataset,3,14,1,0,5,0.176471,0.750000,0.285714,0.6,0.0
2,spatial_range_km2,1,0,3,1,5,1.000000,0.250000,0.400000,0.4,0.4
6,species,6,20,3,0,5,0.230769,0.666667,0.342857,1.2,0.0
5,temp_range_f,3,1,2,0,5,0.750000,0.600000,0.666667,0.6,0.6
4,temp_range_i,4,0,1,0,5,1.000000,0.800000,0.888889,0.8,0.8
3,temporal_range,0,4,5,0,5,0.000000,0.000000,NaN,0.0,0.0


In [13]:
# Aggregate metrics
micro = micro_average(report.field_metrics.values())
macro = macro_f1(report.field_metrics.values())

print("=" * 50)
print("AGGREGATE METRICS")
print("=" * 50)
print(f"Micro-average Precision: {micro.precision:.3f}" if micro.precision else "Micro Precision: N/A")
print(f"Micro-average Recall:    {micro.recall:.3f}" if micro.recall else "Micro Recall: N/A")
print(f"Micro-average F1:        {micro.f1:.3f}" if micro.f1 else "Micro F1: N/A")
print(f"Macro-average F1:        {macro:.3f}" if macro else "Macro F1: N/A")
print("=" * 50)

AGGREGATE METRICS
Micro-average Precision: 0.296
Micro-average Recall:    0.538
Micro-average F1:        0.382
Macro-average F1:        0.491


In [14]:
# Detailed per-record comparison results
print("Per-record, per-field results (mismatches highlighted):")
results_df = report.to_pandas()
mismatches = results_df[~results_df['match']]
print(f"\nTotal mismatches: {len(mismatches)}")
display(mismatches)

Per-record, per-field results (mismatches highlighted):

Total mismatches: 26


,record_id,field,true_value,pred_value,match,tp,fp,fn,tn
0,10.5061/dryad.2n5h6,data_type,"[presence-only, genetic_analysis]","[genetic_analysis, time_series, distribution]",False,1,2,1,0
1,10.5061/dryad.2n5h6,geospatial_info_dataset,[sample],"[sample, site, administrative_units]",False,1,2,0,0
2,10.5061/dryad.2n5h6,spatial_range_km2,2080.0,None,False,0,0,1,0
3,10.5061/dryad.2n5h6,temporal_range,2011-2014,2011 to 2014,False,0,1,1,0
6,10.5061/dryad.2n5h6,species,[black-legged tick],"[black-legged tick, Lyme disease vector, Ixode...",False,1,3,0,0
7,10.5061/dryad.3nh72,data_type,"[presence-only, genetic_analysis]","[genetic_analysis, distribution]",False,1,1,1,0
8,10.5061/dryad.3nh72,geospatial_info_dataset,[sample],"[sample, site_ids, geographic_features, admini...",False,1,4,0,0
10,10.5061/dryad.3nh72,temporal_range,2010-2014,2010\t\t6\t\t14,False,0,1,1,0
13,10.5061/dryad.3nh72,species,"[Eastern coyote, eastern wolf]","[Canis lupus familiaris, Canis lycaon, eastern...",False,1,11,1,0
14,10.5061/dryad.4767v,data_type,[abundance],"[abundance, distribution]",False,1,1,0,0


---

## Results Synthesis & Analysis

### Overall Performance
| Metric | Value |
|--------|-------|
| Micro-average Precision | 0.296 |
| Micro-average Recall | 0.538 |
| Micro-average F1 | 0.382 |
| Macro-average F1 | 0.491 |

### Per-Field Analysis

**Strong Performance:**
- **Temporal year fields** (`temp_range_i`, `temp_range_f`): F1 = 0.89 (start year), 0.67 (end year) — The model reliably extracts start/end years when explicitly mentioned in abstracts. Perfect precision (1.0) for start year with 80% recall.

**Moderate Performance:**
- **Spatial range** (`spatial_range_km2`): Precision = 1.0, Recall = 0.25, F1 = 0.40 — When the model extracts a value, it's correct, but it often misses values (conservative extraction, only 1/4 values extracted).
- **Data type**: Precision = 0.27, Recall = 0.57, F1 = 0.36 — The model tends to over-extract EBV categories (11 FP vs 4 TP), suggesting it identifies more data types than annotators. Reasonable recall indicates it captures most true categories.
- **Species**: Precision = 0.23, Recall = 0.67, F1 = 0.34 — High recall but poor precision. The model extracts many taxonomic terms (20 FP vs 6 TP), over-extracting compared to manual annotations.

**Weak Performance:**
- **Temporal range** (text): F1 = NaN — Complete failure in string matching (0 TP, 4 FP, 5 FN) because manual annotations use different phrasing than model output (e.g., "2008-2012" vs "from 2008 to 2012").
- **Geospatial info**: Precision = 0.18, Recall = 0.75, F1 = 0.29 — High FP count (14) suggests over-prediction of geospatial categories, though recall is decent.

### Key Issues Identified

1. **Vocabulary Mismatch**: Manual annotations use free-text or comma-separated values (e.g., "presence only, EBV genetic analysis") while the schema uses strict enums. Normalization is incomplete.

2. **Over-extraction**: The model tends to identify more categories than annotators (especially for `data_type` and `geospatial_info_dataset`), leading to high FP rates.

3. **String vs Semantic Matching**: Fields like `temporal_range` fail exact matching despite containing equivalent information (e.g., "2008-2012" vs "from 2008 to 2012").

4. **Annotation Granularity**: Manual annotations may have been made from full article/dataset context, while model only sees the abstract.

---

## Conclusion

The current pipeline successfully implements end-to-end feature extraction and evaluation. However, with a **Micro F1 of ~0.40**, there is significant room for improvement before production use.

The evaluation framework (`evaluation.py`) works correctly for comparing Pydantic models, but the gap between manual annotation conventions and model output vocabulary creates artificial mismatches.

---

## Next Steps

### Immediate (Normalization)
1. **Standardize `data_type` vocabulary**: Create mapping from manual annotation strings to `EBVDataType` enum values
2. **Implement fuzzy matching** for `temporal_range` and `species` fields
3. **Review `geospatial_info_dataset` categories**: The model may be correctly inferring info not explicitly annotated

### Short-term (Evaluation)
4. **Expand test set**: Run on all 11 abstract-annotated Dryad records
5. **Add confidence intervals**: Bootstrap the metrics to assess reliability
6. **Create confusion matrices** per field to understand error patterns

### Medium-term (Model Improvement)
7. **Prompt engineering**: Refine system message to align output with annotation guidelines
8. **Few-shot examples**: Add representative examples to the prompt
9. **Schema refinement**: Consider whether `DatasetFeatureExtraction` fields match annotation practice

### Long-term (Pipeline)
10. **Multi-source extraction**: Combine abstract + repository page + article for fuller context
11. **Human-in-the-loop**: Flag low-confidence extractions for manual review
12. **Active learning**: Use mismatches to improve model iteratively

### Understanding the Metrics

**Core Metrics:**
- **Precision**: Of all values the model extracted, what fraction were correct? High precision = few false positives (model doesn't over-extract).
- **Recall**: Of all true values that exist, what fraction did the model find? High recall = few false negatives (model doesn't miss things).
- **F1 Score**: Harmonic mean of precision and recall (balances both). Best when both are high.

**Confusion Matrix Terms:**
- **TP (True Positives)**: Model correctly extracted values that match manual annotations
- **FP (False Positives)**: Model extracted values that don't match manual annotations (over-extraction)
- **FN (False Negatives)**: Manual annotations that the model missed (under-extraction)
- **TN (True Negatives)**: Cases where both agreed on absence (less relevant for extraction tasks)

**Aggregate Metrics:**
- **Micro-average**: Pools all TP/FP/FN across fields, then computes metrics (emphasizes high-volume fields)
- **Macro-average F1**: Averages F1 scores across all fields (treats each field equally, regardless of volume)

**Interpretation for This Task:**
- High precision, low recall → Model is conservative (extracts only when very confident)
- Low precision, high recall → Model is aggressive (extracts liberally but with errors)
- F1 near 0.40 suggests the model captures meaningful information but with substantial noise
- Field-specific F1 variance shows some features are easier to extract than others